<a href="https://colab.research.google.com/github/rudrarapubhavani/Bhavani/blob/main/Data_Cleaning_Transformation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
import pandas as pd
import os

# Download the dataset
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

# List files
print("Downloaded files:", os.listdir(path))

!cp -r {path}/* /content/
from pyspark.sql import SparkSession
from pyspark.sql.functions import *


Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Downloaded files: ['olist_customers_dataset.csv', 'olist_sellers_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_order_items_dataset.csv', 'olist_products_dataset.csv', 'olist_geolocation_dataset.csv', 'product_category_name_translation.csv', 'olist_orders_dataset.csv', 'olist_order_payments_dataset.csv']


In [2]:
spark= SparkSession.builder.appName('Olive_set').getOrCreate()
path='/content/'


In [3]:
customer_df= spark.read.csv(path+'olist_customers_dataset.csv', header= True, inferSchema= True)
geolocation_df=spark.read.csv(path+'olist_geolocation_dataset.csv', header= True, inferSchema= True)
order_items_df= spark.read.csv(path+'olist_order_items_dataset.csv', header= True, inferSchema= True)
order_payments_df= spark.read.csv(path+'olist_order_payments_dataset.csv', header= True, inferSchema= True)
order_review_df= spark.read.csv(path+'olist_order_reviews_dataset.csv', header= True, inferSchema= True)
orders_df= spark.read.csv(path+'olist_orders_dataset.csv', header= True, inferSchema= True)
product_df= spark.read.csv(path+'olist_products_dataset.csv', header= True, inferSchema= True)
sellers_df= spark.read.csv(path+'olist_sellers_dataset.csv', header=True, inferSchema= True)
product_category= spark.read.csv(path+'product_category_name_translation.csv', header= True, inferSchema= True)


In [4]:
customer_df.createOrReplaceTempView('customer')
geolocation_df.createOrReplaceTempView('geolocation')
order_items_df.createOrReplaceTempView('order_item')
order_payments_df.createOrReplaceTempView('order_payment')
order_review_df.createOrReplaceTempView('order_review')
orders_df.createOrReplaceTempView('order')
product_df.createOrReplaceTempView('product')
sellers_df.createOrReplaceTempView('seller')
product_category.createOrReplaceTempView('product_category')


# Identify Missing Values

In [11]:
#customer table

feilds= ','.join([f'sum( case when {c} is null then 1 else 0 end ) as {c}' for c in customer_df.columns])
print('customer table')
spark.sql(
   f'''select {feilds} from customer
    '''
).show()

#geoLocation table

print('Geo Location table')
feilds= ','.join([f'sum( case when {c} is null then 1 else 0 end ) as {c}' for c in geolocation_df.columns])
spark.sql(
   f'''select {feilds} from geolocation
    '''
).show()

# order item table


feilds= ','.join([f'sum( case when {c} is null then 1 else 0 end ) as {c}' for c in order_items_df.columns])
print('order table')
spark.sql(
   f'''select {feilds} from order_item
    '''
).show()

# order payments table

feilds= ','.join([f'sum( case when {c} is null then 1 else 0 end ) as {c}' for c in order_payments_df.columns])
print('order payments table')
spark.sql(
   f'''select {feilds} from order_payment
    '''
).show()

# oder review table

feilds= ','.join([f'sum( case when {c} is null then 1 else 0 end ) as {c}' for c in order_review_df.columns])
print('order review table')
spark.sql(
   f'''select {feilds} from order_review
    '''
).show()


# order table

feilds= ','.join([f'sum( case when {c} is null then 1 else 0 end ) as {c}' for c in orders_df.columns])
print('order table')
spark.sql(
   f'''select {feilds} from order
    '''
).show()

# Product table

feilds= ','.join([f'sum( case when {c} is null then 1 else 0 end ) as {c}' for c in product_df.columns])
print('product table')
spark.sql(
   f'''select {feilds} from product
    '''
).show()

# seller table

feilds= ','.join([f'sum( case when {c} is null then 1 else 0 end ) as {c}' for c in sellers_df.columns])
print('seller table')
spark.sql(
   f'''select {feilds} from seller
    '''
).show()

# product category

feilds= ','.join([f'sum( case when {c} is null then 1 else 0 end ) as {c}' for c in product_category.columns])
print('product category')
spark.sql(
   f'''select {feilds} from product_category
    '''
).show()


customer table
+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+

Geo Location table
+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|                          0|              0|              0|               0|                0|
+---------------------------+---------------+---------------+----------------+-----------------+

order table
+--------+-------------+--------

#order review have missing values in all columns



In [12]:
order_review_df.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)



In [13]:
spark.sql(
    'select * from order_review limit(5)'
).show()

+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|7bc2406110b926393...|73fc7af87114b3971...|           4|                NULL|                  NULL| 2018-01-18 00:00:00|    2018-01-18 21:46:59|
|80e641a11e56f04c1...|a548910a1c6147796...|           5|                NULL|                  NULL| 2018-03-10 00:00:00|    2018-03-11 03:05:13|
|228ce5500dc1d8e02...|f9e4b658b201a9f2e...|           5|                NULL|                  NULL| 2018-02-17 00:00:00|    2018-02-18 14:36:24|
|e64fb393e7b32834b...|658677c97b385a9be...|           5|                NULL|  Recebi bem antes ...| 2017-04-21 00:00:00|   

In [16]:
order_review_cleaned_df=spark.sql(
    '''select review_id, order_id, review_score,
    coalesce(review_comment_title, 'No Title Provided'),
    coalesce(review_comment_message, 'No Message Provided'),
    review_creation_date,
    review_answer_timestamp
    from order_review
    '''
)

In [18]:
order_review_cleaned_df=spark.sql(
    '''select * from order_review
    where review_id is not null
    '''
)

In [20]:
order_review_cleaned_df.createOrReplaceTempView('order_review')

# Handling Missing Values for Order Table



In [21]:
cleaned_orders= spark.sql(
    '''
    select order_id,order_status,
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date
    from order
    where order_status == 'delivered' and
    order_delivered_carrier_date is not null and
    order_estimated_delivery_date is not null



    '''
)



In [27]:
in_process_order = spark.sql('''
    SELECT * FROM order
    WHERE order_status NOT IN ('delivered', 'canceled', 'unavailable')
''')

In [23]:
cleaned_orders.createOrReplaceTempView('order_cleaned')

In [45]:
imputed_orders= spark.sql(
    '''
    select order_id,order_status,customer_id,
    coalesce(order_approved_at,order_purchase_timestamp) as order_approved_at,

    order_delivered_carrier_date,
    order_estimated_delivery_date,
    case
    when order_delivered_customer_date is null then 'In-progress/ In-Complete'
    else 'Fully Delivered' end as logistic_status
    from order
    '''
)

In [46]:
imputed_orders.createOrReplaceTempView('orders_cleaned')

In [47]:
spark.sql('''
select * from orders_cleaned limit(5)''').show()

+--------------------+------------+--------------------+-------------------+----------------------------+-----------------------------+---------------+
|            order_id|order_status|         customer_id|  order_approved_at|order_delivered_carrier_date|order_estimated_delivery_date|logistic_status|
+--------------------+------------+--------------------+-------------------+----------------------------+-----------------------------+---------------+
|e481f51cbdc54678b...|   delivered|9ef432eb625129730...|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-18 00:00:00|Fully Delivered|
|53cdb2fc8bc7dce0b...|   delivered|b0830fb4747a6c6d2...|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-13 00:00:00|Fully Delivered|
|47770eb9100c2d0c4...|   delivered|41ce2a54c0b03bf34...|2018-08-08 08:55:23|         2018-08-08 13:50:00|          2018-09-04 00:00:00|Fully Delivered|
|949d5b44dbf5de918...|   delivered|f88197465ea7920ad...|2017-11-18 19:45:59|         201

# Creating Sales table from above oders, payment,product,customer tables

In [79]:
# Removing Duplicates

Cleaned_Olive_data1=spark.sql(
    '''
    select o.order_id,o.order_status,o.order_delivered_carrier_date,
    o.order_estimated_delivery_date,
    c.customer_id,c.customer_unique_id,c.customer_state, c.customer_city,
    oi.seller_id,oi.price,oi.freight_value,oi.shipping_limit_date,

    op.payment_installments,prod.product_category_name,
    op.payment_value as total_payment,op.payment_type,
    prod.product_weight_g
    from
    orders_cleaned o
    inner join customer c on c.customer_id= o.customer_id
    inner join order_item oi on oi.order_id= o.order_id
    inner join order_payment op on op.order_id= o.order_id
    inner join product prod on oi.product_id = prod.product_id

    left join(
    select order_id, sum(payment_value) as payment
    from order_payment group by order_id) as p on o.order_id = p.order_id

    '''
)

In [80]:
Cleaned_Olive_data1.createOrReplaceTempView('cleaner_oliset_data')

In [84]:
spark.sql('select * from cleaner_oliset_data limit(5)').show()

+--------------------+------------+----------------------------+-----------------------------+--------------------+--------------------+--------------+-------------+--------------------+------+-------------+-------------------+--------------------+---------------------+-------------+------------+----------------+-------------------+
|            order_id|order_status|order_delivered_carrier_date|order_estimated_delivery_date|         customer_id|  customer_unique_id|customer_state|customer_city|           seller_id| price|freight_value|shipping_limit_date|payment_installments|product_category_name|total_payment|payment_type|product_weight_g|Product_weight_tier|
+--------------------+------------+----------------------------+-----------------------------+--------------------+--------------------+--------------+-------------+--------------------+------+-------------+-------------------+--------------------+---------------------+-------------+------------+----------------+----------------

# Advanced Transformation

In [81]:
Cleaned_Olive_data1= spark.sql(
    '''
    select * from cleaner_oliset_data where product_weight_g >= 10 and
    product_weight_g <=3000
    '''
)

Cleaned_Olive_data1.createOrReplaceTempView('cleaner_oliset_data')

In [82]:
Cleaned_Olive_data1= spark.sql(
    '''
    select *, case
    when
    product_weight_g < 1000 then 'small'
    when product_weight_g > 1000 and product_weight_g< 5000 then 'medium'
    else 'Large' end
    as Product_weight_tier

    from cleaner_oliset_data

    '''
)

Cleaned_Olive_data1.createOrReplaceTempView('cleaner_oliset_data')

In [91]:
Cleaned_Olive_data1=spark.sql(
    '''
    select *,


    COUNT(order_id) OVER(PARTITION BY order_id) AS total_orders,


   ROUND( AVG(total_payment) OVER(PARTITION BY order_id), 2) AS avg_payment,
  ROUND(SUM(total_payment) OVER(PARTITION BY order_id), 2) AS total_tier_revenue



    from cleaner_oliset_data

    order by total_tier_revenue

    '''
)

In [92]:
Cleaned_Olive_data1.createOrReplaceTempView('cleaner_oliset_data')

In [95]:
spark.sql('select * from cleaner_oliset_data limit(5)').show()

Cleaned_Olive_data1.printSchema()

+--------------------+------------+----------------------------+-----------------------------+--------------------+--------------------+--------------+-------------+--------------------+-----+-------------+-------------------+--------------------+---------------------+-------------+------------+----------------+-------------------+------------+-----------+------------------+
|            order_id|order_status|order_delivered_carrier_date|order_estimated_delivery_date|         customer_id|  customer_unique_id|customer_state|customer_city|           seller_id|price|freight_value|shipping_limit_date|payment_installments|product_category_name|total_payment|payment_type|product_weight_g|Product_weight_tier|total_orders|avg_payment|total_tier_revenue|
+--------------------+------------+----------------------------+-----------------------------+--------------------+--------------------+--------------+-------------+--------------------+-----+-------------+-------------------+------------------

In [98]:
spark.sql(
    '''
create external table if not exists cleaned_order_parquet (
    order_id string,
    order_status string,
    order_delivered_carrier_date timestamp ,
 order_estimated_delivery_date timestamp ,
  customer_id string ,
  customer_unique_id string ,
  customer_state string ,
  customer_city string ,
 seller_id string ,
  price double ,
  freight_value double ,
  shipping_limit_date timestamp ,
 payment_installments integer ,
 product_category_name string ,
 total_payment double ,
 payment_type string ,
 product_weight_g integer ,
 Product_weight_tier string ,
 total_orders long ,
avg_payment double ,
total_tier_revenue double

) stored as parquet location '/content/spark-warehouse'

    '''
)

AnalysisException: [NOT_SUPPORTED_COMMAND_WITHOUT_HIVE_SUPPORT] CREATE Hive TABLE (AS SELECT) is not supported, if you want to enable it, please set "spark.sql.catalogImplementation" to "hive". SQLSTATE: 0A000;
'CreateTable `spark_catalog`.`default`.`cleaned_order_parquet`, org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe, Ignore


In [101]:
# 1. Wrap the smaller DataFrame with the broadcast function
customer_broadcast_df = broadcast(Cleaned_Olive_data1)

# 2. Perform the optimized join natively using standard PySpark syntax
joined_df = Cleaned_Olive_data1.join(customer_broadcast_df, on='customer_id', how='inner')

# Display the result
joined_df.show()


+--------------------+--------------------+------------+----------------------------+-----------------------------+--------------------+--------------+--------------------+--------------------+------+-------------+-------------------+--------------------+---------------------+-------------+------------+----------------+-------------------+------------+-----------+------------------+--------------------+------------+----------------------------+-----------------------------+--------------------+--------------+--------------------+--------------------+------+-------------+-------------------+--------------------+---------------------+-------------+------------+----------------+-------------------+------------+-----------+------------------+
|         customer_id|            order_id|order_status|order_delivered_carrier_date|order_estimated_delivery_date|  customer_unique_id|customer_state|       customer_city|           seller_id| price|freight_value|shipping_limit_date|payment_installme

In [105]:
from google.colab import files

# 1. Take a 100-row sample of your optimized dataframe
sample_df = Cleaned_Olive_data1

# 2. Convert to a standard Pandas DataFrame
pandas_sample = sample_df.toPandas()

# 3. Save it as a CSV file in your local Colab session files
pandas_sample.to_csv('/content/cleaned_order_sample.csv', index=False)

# 4. Trigger an immediate browser download to your computer
files.download('/content/cleaned_order_sample.csv')

print("Check your computer's Downloads folder!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Check your computer's Downloads folder!
